# 🏗️ Lakehouse Data Platform — Data Mesh Pipeline

## Business Scenario

A retail company operates a **lakehouse data platform** with domain-driven ownership.
Multiple domain teams manage their own data pipelines independently:

| Domain | System | Data Products |
|--------|--------|---------------|
| **CRM** ← *this example* | Webshop | Customers, Orders, Revenue Analytics |
| Sales | POS, Marketplace | Transactions, Returns, Commissions |
| Marketing | Google Analytics, CMS | Events, Sessions, Channel Performance |
| Finance | ERP, Billing | Invoices, Payments, GL Entries |
| Shared / Reference | MDM | Countries, Currencies, Product Catalog |

Every domain follows the **same standardized process** — powered by LakeLogic data contracts:

### Why Data Mesh with LakeLogic?

| Benefit | How |
|---------|-----|
| 🚀 **Fast onboarding** | New data sources onboarded by adding a contract YAML — no code, no central team bottleneck |
| 👥 **Domain autonomy** | Each domain team owns their schemas, quality rules, and PII masking — no central team dependency |
| 🔒 **Consistent governance** | System-level config (`_system.yaml`) enforces standards (lineage, Delta format, schema evolution) across all contracts |
| ⚡ **Engine portability** | Prototype with DuckDB locally, deploy the same contracts on Spark/Databricks — zero contract changes |

> 💡 *More capabilities (test data generation, bootstrap onboarding, external logic, unstructured data) are covered in [Advanced Capabilities](#advanced-capabilities) below.*

---

### Configuration Hierarchy

LakeLogic contracts inherit from a three-level config hierarchy — defaults flow down, overrides flow up:

| Level | File | Scope |
|-------|------|-------|
| **Domain** | `_domain.yaml` | SLOs, governance policies, retention, PII policy — applies to all systems within the domain |
| **System** | `_system.yaml` | Materialization, lineage, schema evolution — applies to all contracts |
| **Contract** | `*_v1.0.yaml` | Schema, quality rules, PII masking — can override domain/system defaults |

---

### Folder Structure

```
04_lakehouse_data_platform/                     ← company data platform repo
├── contracts/
│   └── crm/                                    ← domain (business unit)
│       ├── _domain.yaml                        ← domain-level config (SLOs)
│       └── webshop/                            ← system (source)
│           ├── _system.yaml                    ← system-level config
│           ├── bronze/                         ← medallion layer
│           │   ├── bronze_customers_v1.0.yaml
│           │   └── bronze_orders_v1.0.yaml
│           ├── silver/
│           │   ├── silver_customers_v1.0.yaml
│           │   └── silver_orders_enriched_v1.0.yaml
│           └── gold/
│               ├── gold_dim_customers_v1.0.yaml
│               └── gold_daily_revenue_v1.0.yaml
├── landing_crm/webshop/                        ← landing zone
└── lakehouse/                                  ← Delta Lake output
    └── crm/                                    ← domain schema
        ├── bronze_webshop_customers/
        ├── bronze_webshop_orders/
        ├── silver_webshop_customers/
        ├── silver_webshop_orders_enriched/
        ├── gold_webshop_dim_customers/
        └── gold_webshop_daily_revenue/
```

---

### Architecture

```mermaid
graph TD
    API["Webshop API"] --> LC["Landing Zone"]
    LC --> B["🥉 Bronze"]
    B --> S["🥈 Silver"]
    S --> G["🥇 Gold"]
    G --> DL["Lakehouse"]

    Gov["⚙️ LakeLogic"]
    Gov -.-> B
    Gov -.-> S
    Gov -.-> G
```

---

### Engine Portability — This Notebook is the Proof

This entire notebook runs on **DuckDB** — a single-process analytical engine. The key insight:
**every contract YAML you see below deploys unchanged on Spark/Databricks in production.**

| Phase | Engine | Use Case |
|-------|--------|----------|
| 🧪 **Prototype** ← *you are here* | DuckDB | Instant feedback in a notebook |
| 🔬 **Develop** | Polars | Larger local datasets, CI/CD testing |
| 🏭 **Production** | Spark / Databricks | Cluster-scale, Unity Catalog, scheduled DABs |

Switch with `ENGINE = 'duckdb'` → `'polars'` → `'spark'`. Zero contract changes.

---

<a id="advanced-capabilities"></a>

### Advanced Capabilities

Beyond the core Data Mesh fundamentals above, LakeLogic provides:

| Capability | How |
|------------|-----|
| 📦 **Standardized process** | Every domain uses the same Bronze → Silver → Gold pipeline pattern — reducing learning curves and ops overhead |
| 🔄 **Contract overrides** | Individual contracts can override system defaults when needed (e.g. custom `target_path`, `strategy: overwrite`) |
| 🧪 **Test data generation** | Auto-generate realistic test data from contracts — supports AI/LLM Bring Your Own Model for domain-accurate synthetic data |
| 📋 **Bootstrap onboarding** | Auto-generate contracts from existing schemas (dbt, database, CSV) — fast onboarding of legacy datasets |
| 🔌 **External logic** | Call external Python scripts or notebooks for complex processing — sandboxed execution with automatic input/output handoff via `external_logic:` |
| 📄 **Unstructured data** | Process unstructured data (PDFs, emails, logs) via contract-defined LLM extraction pipelines |

---
## 1. 📦 Setup

Install LakeLogic with DuckDB support. Same package, same contracts — `pip install lakelogic[spark]`
on Databricks for production.

In [45]:
%%capture
%pip install lakelogic[duckdb, ai] -q

In [46]:
import os
import sys
import subprocess

# Editable install for local development (skip on Colab)
_LOCAL = os.path.abspath(os.path.join(os.path.dirname("__file__"), "..", ".."))
if os.path.isfile(os.path.join(_LOCAL, "pyproject.toml")):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", _LOCAL, "-q"])
    print(f"\u2705 Installed lakelogic[duckdb, ai] (editable) from {_LOCAL}")
else:
    print("\u2705 Using lakelogic from pip")

✅ Installed lakelogic[duckdb, ai] (editable) from c:\_Personal\_SaaS\lakelogic


In [47]:
from pathlib import Path

# ── Resolve paths relative to this notebook ─────────────────────
EXAMPLE_ROOT = Path(".").resolve()
SYSTEM_YAML = EXAMPLE_ROOT / "contracts" / "crm" / "webshop" / "_system.yaml"

# Engine configuration
ENGINE = "duckdb"  # Full DuckDB engine for transforms + quality
STORAGE_MODE = "direct"  # Local file paths, no Unity Catalog

print(f"\u2705 Example root:  {EXAMPLE_ROOT}")
print(f"\u2705 System YAML:   {SYSTEM_YAML}")
print(f"\u2705 Engine:        {ENGINE}")
print(f"\u2705 Storage mode:  {STORAGE_MODE}")

✅ Example root:  C:\_Personal\_SaaS\lakelogic\examples\04_lakehouse_data_platform
✅ System YAML:   C:\_Personal\_SaaS\lakelogic\examples\04_lakehouse_data_platform\contracts\crm\webshop\_system.yaml
✅ Engine:        duckdb
✅ Storage mode:  direct


---
## 2. 🧪 Generate Synthetic Landing Data

Simulating the webshop API landing raw CSV files into `landing_crm/webshop/`.
In production, this would be an ingestion pipeline writing to ADLS / S3.

LakeLogic's `DataGenerator` creates realistic test data directly from contracts.
For even more realistic data, provide your own AI/LLM model (see config below).

In [48]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

# ── AI/LLM Config (optional) ──────────────────────────────────
# Provide your own model for more realistic synthetic data.
# Uncomment and set your API key + model to enable.
# AI_CONFIG = {
#     'provider': 'anthropic',                    # openai | azure | anthropic | google | ollama
#     'api_key': '',                     # or set OPENAI_API_KEY env var
#     'model': 'claude-haiku-4-5',                  # model name e.g gpt-4o-mini, gemini-2.0-flash, claude-haiku-4-5
#     'base_url': None,                        # custom endpoint (Azure, Ollama)
# }
AI_CONFIG = None  # Set to dict above to enable AI-generated test data

from lakelogic.core.generator import DataGenerator
import datetime
import polars as pl
from pathlib import Path
import random


CONTRACTS = {
    "customers": EXAMPLE_ROOT / "contracts" / "crm" / "webshop" / "bronze" / "bronze_customers_v1.0.yaml",
    "orders": EXAMPLE_ROOT / "contracts" / "crm" / "webshop" / "bronze" / "bronze_orders_v1.0.yaml",
}

# Landing zone: landing_{domain}/{system}/{dataset}
LANDING_ROOT = EXAMPLE_ROOT / "landing_crm" / "webshop"

ROWS_PER_PARTITION = 7
NUM_DAYS = 2
today = datetime.date.today()
random_int = random.randint(38, 45)

# AI config passthrough
ai_kwargs = {}
if AI_CONFIG:
    import os

    if AI_CONFIG.get("api_key"):
        provider = AI_CONFIG.get("provider", "openai")
        env_keys = {
            "openai": "OPENAI_API_KEY",
            "azure": "AZURE_OPENAI_API_KEY",
            "anthropic": "ANTHROPIC_API_KEY",
            "google": "GOOGLE_API_KEY",
        }
        env_var = env_keys.get(provider, f"{provider.upper()}_API_KEY")
        os.environ[env_var] = AI_CONFIG["api_key"]
    ai_kwargs = {
        "ai": True,
        "ai_provider": AI_CONFIG.get("provider"),
        "ai_model": AI_CONFIG.get("model"),
    }

for day_offset in range(NUM_DAYS):
    day = today - datetime.timedelta(days=day_offset)

    # generate_related() detects FK relationships and generates
    # customers first → orders with valid customer_ids (no manual sync)
    related = DataGenerator.generate_related(
        contracts={k: str(v) for k, v in CONTRACTS.items()},
        rows=ROWS_PER_PARTITION,
        invalid_ratio=0.02,
        seed=random_int + day_offset,
        **ai_kwargs,
    )

    for entity, df in related.items():
        partition = LANDING_ROOT / entity / day.strftime("y_%Y/m_%m/d_%d")
        partition.mkdir(parents=True, exist_ok=True)
        df.write_csv(str(partition / "data.csv"))

for entity in CONTRACTS:
    total_files = sum(1 for _ in (LANDING_ROOT / entity).rglob("*.csv"))
    print(f"{entity}: {total_files} CSV files across {NUM_DAYS} partitions")

print(f" Landing zone: {LANDING_ROOT}")

orders_dir = LANDING_ROOT / "orders"
cust_dir = LANDING_ROOT / "customers"
print(" Sample orders:")
display(pl.read_csv(str(next(orders_dir.rglob("*.csv")))).sort("customer_id").head(10))
print(" Sample customers (PII fields: name, email, phone):")
display(pl.read_csv(str(next(cust_dir.rglob("*.csv")))).sort("customer_id").head(10))

2026-04-09 06:32:27.808 | INFO     | lakelogic.core.generator:generate_related:3679 - 🔗 Detected FK: orders.customer_id → customers.customer_id
2026-04-09 06:32:27.809 | INFO     | lakelogic.core.generator:generate_related:3707 - 📋 Generation order: customers → orders
2026-04-09 06:32:27.811 | INFO     | lakelogic.core.generator:generate:3022 - 📋 Generating data for: Webshop Customers (Bronze)
2026-04-09 06:32:27.812 | INFO     | lakelogic.core.generator:generate:3023 -    Records    : 7 valid + 0 invalid = 7 total
2026-04-09 06:32:27.812 | INFO     | lakelogic.core.generator:generate:3036 -    Source     : Faker + heuristic generation (no AI or file seeds)
2026-04-09 06:32:27.819 | INFO     | lakelogic.core.generator:generate:3070 -    ✅ Row generation complete: 7 records built
2026-04-09 06:32:27.826 | INFO     | lakelogic.core.generator:generate_related:3736 -    orders.customer_id ← 7 unique values from customers.customer_id
2026-04-09 06:32:27.828 | INFO     | lakelogic.core.gener

customers: 2 CSV files across 2 partitions
orders: 2 CSV files across 2 partitions
 Landing zone: C:\_Personal\_SaaS\lakelogic\examples\04_lakehouse_data_platform\landing_crm\webshop
 Sample orders:


order_id,customer_id,order_date,product_name,category,quantity,unit_price,discount_pct,channel,status,_is_invalid
str,str,str,str,str,i64,f64,f64,str,str,bool
"""ORD-355280""","""CUST-122317""","""2026-02-14""","""Mounting Bracket""","""Clothing""",4424,4.77,35.0,"""in-store""","""active""",false
"""ORD-961429""","""CUST-136360""","""2026-02-02""","""Power Supply 500W""","""Sports""",712,43.43,730.0,"""partner""","""active""",false
"""ORD-428381""","""CUST-166775""","""2026-03-19""","""Control Board v2""","""Sports""",3682,120.45,387.0,"""partner""","""active""",false
"""ORD-511340""","""CUST-428381""","""2026-02-14""","""Circuit Board""","""Health""",7370,25.36,666.0,"""social""","""active""",false
"""ORD-424182""","""CUST-428381""","""2026-02-24""","""Connector Kit""","""Sports""",972,60.58,146.0,"""online""","""active""",false
"""ORD-307966""","""CUST-772849""","""2026-01-17""","""Cooling Fan 120mm""","""Health""",8581,37.7,268.0,"""partner""","""active""",false
"""ORD-493752""","""CUST-772849""","""2026-03-08""","""Power Supply 500W""","""Food""",5668,70.25,569.0,"""partner""","""active""",false


 Sample customers (PII fields: name, email, phone):


customer_id,name,email,phone,country,segment,signup_date,_is_invalid
str,str,str,str,str,str,str,bool
"""CUST-122317""","""Gerald Campbell""","""danieljones@example.org""","""451-640-5617x733""","""South Africa""","""win_back""","""2026-02-26""",false
"""CUST-136360""","""Joseph Gregory""","""ramirezsusan@example.com""","""689-858-7147""","""United Kingdom""","""dormant""","""2026-04-07""",false
"""CUST-166775""","""Daniel Smith""","""virginia59@example.org""",null,"""France""","""vip""","""2026-03-30""",false
"""CUST-428381""","""Joshua Atkins""","""noahroberts@example.org""","""890-712-8469""","""France""","""churned""","""2026-02-06""",false
"""CUST-619706""","""Paula Williams""","""vmckinney@example.net""","""001-255-677-9823x775""","""Italy""","""vip""","""2026-03-29""",false
"""CUST-772849""","""Joshua Bender""","""stewartdavid@example.org""","""+1-721-576-5714x92488""","""Spain""","""high_value""","""2026-03-19""",false
"""CUST-861748""","""Isaac Rodriguez""","""catherinebrown@example.com""","""001-809-671-2209x41410""","""Netherlands""","""high_value""","""2026-02-02""",false


---
## 3. 🔧 Load System Registry & Initialize Pipeline

The `_system.yaml` inside `contracts/crm/` provides system-level defaults:
- **Materialization**: Delta format, append/merge strategies per layer
- **Lineage**: Automatic `_lakelogic_processed_at` and `_lakelogic_run_id` columns
- **Schema evolution**: Bronze allows drift, Silver/Gold enforce strict schemas

Each contract inherits these defaults but can override when needed.

In [49]:
from lakelogic.core.registry import DomainRegistry
from lakelogic.pipeline import LakehousePipeline

registry = DomainRegistry.from_yaml(str(SYSTEM_YAML), storage_mode=STORAGE_MODE)

pipeline = LakehousePipeline(registry, engine=ENGINE, spark=None)

print("\u2705 Pipeline initialized")
print(f"  Run ID:     {pipeline.run_id}")
print(f"  Contracts:  {len(registry.contracts)}")
print(f"  Layers:     {sorted(set(c.layer for c in registry.contracts))}")

✅ Pipeline initialized
  Run ID:     882287ee-2a2b-4035-8872-c7e7fcb9c3a2
  Contracts:  6
  Layers:     ['bronze', 'gold', 'silver']


---
## 4. 🗺️ Pipeline DAG

The DAG shows how data products within the CRM domain interact:
- **Within a data product**: Bronze → Silver → Gold flows
- **Cross-product**: Orders enriched with customer context via `links:` dependency

In [50]:
from IPython.display import HTML, display

display(HTML(pipeline.visualize_dag()))

---
## 5. 🥉 Run Bronze — Ingest from Landing Zone

Bronze reads from `landing_crm/webshop/` and applies:
- **Schema enforcement** against the contract definition
- **Quality rules** (valid IDs, positive quantities, valid discounts)
- **PII masking** (name→redact, email→hash, phone→partial)
- **Quarantine** for records that fail validation

All config-driven from the contract — no code changes needed.

In [51]:
summary_bronze = pipeline.run(target_layers="bronze", reset_layers="bronze", created_by="duckdb_demo")

print("\n" + "=" * 80)
print("BRONZE LAYER SUMMARY")
print("=" * 80)
print(f"  {'Table':<45} {'Status':<12} {'Good':>6} {'Bad':>6}")
print(f"  {'-' * 45} {'-' * 12} {'-' * 6} {'-' * 6}")
for r in summary_bronze.results:
    print(
        f"  {r.get('table_name', '?'):<45} {r.get('status', '?'):<12} {str(r.get('rows_good', '-')):>6} {str(r.get('rows_bad', '-')):>6}"
    )
    if r.get("error"):
        print(f"    \u21b3 {r['error']}")
print("=" * 80)

2026-04-09 06:32:28.025 | INFO     | lakelogic.pipeline.runner:run:1138 - Pipeline storage mode: direct
2026-04-09 06:32:28.027 | INFO     | lakelogic.pipeline.runner:_execute_resets:652 - Resetting [bronze] customers
2026-04-09 06:32:28.068 | WARNING  | lakelogic.pipeline.runner:_delete_run_log_entries:642 -   Could not clear run log for 'customers': no log files
2026-04-09 06:32:28.070 | INFO     | lakelogic.pipeline.runner:_execute_resets:652 - Resetting [bronze] orders
2026-04-09 06:32:28.105 | WARNING  | lakelogic.pipeline.runner:_delete_run_log_entries:642 -   Could not clear run log for 'orders': no log files
2026-04-09 06:32:28.111 | INFO     | lakelogic.pipeline.runner:run:1235 - ── Processing Layer: BRONZE (2 contracts) ──
2026-04-09 06:32:28.111 | INFO     | lakelogic.pipeline.runner:_process_single_contract:1317 -   ─────────────────────────────────────────────────────────
2026-04-09 06:32:28.113 | INFO     | lakelogic.pipeline.runner:_process_single_contract:1318 -   📄 [br


BRONZE LAYER SUMMARY
  Table                                         Status         Good    Bad
  --------------------------------------------- ------------ ------ ------
  bronze_webshop_customers                      success          14      0
  bronze_webshop_orders                         success          14      0


---
## 6. 🥈 Run Silver — Cross-Product Enrichment

Orders enriched with customer context via `links:` — a cross-product dependency.
The Order contract declares it needs the Customer data product, LakeLogic resolves it.

Business logic applied:
- Filter out cancelled/returned orders
- JOIN with customer master data (segment, country)
- Compute `line_total = quantity × unit_price × (1 - discount_pct)`

In [52]:
summary_silver = pipeline.run(target_layers="silver", reset_layers="silver", created_by="duckdb_demo")

print("\n" + "=" * 80)
print("SILVER LAYER SUMMARY")
print("=" * 80)
print(f"  {'Table':<45} {'Status':<12} {'Good':>6} {'Bad':>6}")
print(f"  {'-' * 45} {'-' * 12} {'-' * 6} {'-' * 6}")
for r in summary_silver.results:
    print(
        f"  {r.get('table_name', '?'):<45} {r.get('status', '?'):<12} {str(r.get('rows_good', '-')):>6} {str(r.get('rows_bad', '-')):>6}"
    )
    if r.get("error"):
        print(f"    \u21b3 {r['error']}")
print("=" * 80)

2026-04-09 06:32:28.877 | INFO     | lakelogic.pipeline.runner:run:1138 - Pipeline storage mode: direct
2026-04-09 06:32:28.880 | INFO     | lakelogic.pipeline.runner:_execute_resets:652 - Resetting [silver] customers_cleansed
2026-04-09 06:32:28.927 | INFO     | lakelogic.pipeline.runner:_delete_run_log_entries:631 -   Cleared run log entries (dataset=silver_webshop_customers, layer=silver, domain=crm, system=webshop) from lakehouse/crm/_run_logs via delta-rs
2026-04-09 06:32:28.929 | INFO     | lakelogic.pipeline.runner:_execute_resets:652 - Resetting [silver] orders_enriched
2026-04-09 06:32:28.982 | INFO     | lakelogic.pipeline.runner:_delete_run_log_entries:631 -   Cleared run log entries (dataset=silver_webshop_orders_enriched, layer=silver, domain=crm, system=webshop) from lakehouse/crm/_run_logs via delta-rs
2026-04-09 06:32:28.989 | INFO     | lakelogic.pipeline.runner:run:1235 - ── Processing Layer: SILVER (2 contracts) ──
2026-04-09 06:32:28.990 | INFO     | lakelogic.pipel


SILVER LAYER SUMMARY
  Table                                         Status         Good    Bad
  --------------------------------------------- ------------ ------ ------
  silver_webshop_customers                      success          11      3
  silver_webshop_orders_enriched                success           0     14


---
## 7. 🥇 Run Gold — Data Products

Gold tables are the published **data products** consumed by downstream systems:

- **`gold_dim_customers`** → SCD2 dimension → powers Customer Lookup API
- **`gold_daily_revenue`** → aggregated analytics → feeds Power BI dashboard

In [53]:
summary_gold = pipeline.run(target_layers="gold", reset_layers="gold", created_by="duckdb_demo")

print("\n" + "=" * 80)
print("GOLD LAYER SUMMARY")
print("=" * 80)
print(f"  {'Table':<45} {'Status':<12} {'Good':>6} {'Bad':>6}")
print(f"  {'-' * 45} {'-' * 12} {'-' * 6} {'-' * 6}")
for r in summary_gold.results:
    print(
        f"  {r.get('table_name', '?'):<45} {r.get('status', '?'):<12} {str(r.get('rows_good', '-')):>6} {str(r.get('rows_bad', '-')):>6}"
    )
    if r.get("error"):
        print(f"    \u21b3 {r['error']}")
print("=" * 80)

2026-04-09 06:32:30.575 | INFO     | lakelogic.pipeline.runner:run:1138 - Pipeline storage mode: direct
2026-04-09 06:32:30.578 | INFO     | lakelogic.pipeline.runner:_execute_resets:652 - Resetting [gold] dim_customers
2026-04-09 06:32:30.628 | INFO     | lakelogic.pipeline.runner:_delete_run_log_entries:631 -   Cleared run log entries (dataset=gold_webshop_dim_customers, layer=gold, domain=crm, system=webshop) from lakehouse/crm/_run_logs via delta-rs
2026-04-09 06:32:30.629 | INFO     | lakelogic.pipeline.runner:_execute_resets:652 - Resetting [gold] daily_revenue
2026-04-09 06:32:30.689 | INFO     | lakelogic.pipeline.runner:_delete_run_log_entries:631 -   Cleared run log entries (dataset=gold_webshop_daily_revenue, layer=gold, domain=crm, system=webshop) from lakehouse/crm/_run_logs via delta-rs
2026-04-09 06:32:30.693 | INFO     | lakelogic.pipeline.runner:run:1235 - ── Processing Layer: GOLD (2 contracts) ──
2026-04-09 06:32:30.695 | INFO     | lakelogic.pipeline.runner:_process


GOLD LAYER SUMMARY
  Table                                         Status         Good    Bad
  --------------------------------------------- ------------ ------ ------
  gold_webshop_dim_customers                    success          11      0
  gold_webshop_daily_revenue                    no_new_rows       0      0


---
## 8. 🦆 Query the Lakehouse

Query the Delta tables in `lakehouse/` using DuckDB's `delta_scan()` — the same
experience analysts have against trusted, quality-checked, PII-protected data.

In [54]:
import duckdb

con = duckdb.connect()

# ── Discover Delta tables in the lakehouse ──────────────────────
OUTPUT = EXAMPLE_ROOT / "lakehouse"
tables = {}
if OUTPUT.exists():
    for domain_dir in sorted(OUTPUT.iterdir()):
        if domain_dir.is_dir() and not domain_dir.name.startswith("_"):
            print(f"\n📂 {domain_dir.name}/")
            for entry in sorted(domain_dir.iterdir()):
                if entry.is_dir() and (entry / "_delta_log").exists():
                    tables[entry.name] = str(entry).replace("\\", "/")

print(f"\U0001f986 Found {len(tables)} Delta tables in lakehouse/:\n")
for name, path in tables.items():
    try:
        n = con.sql(f"SELECT COUNT(*) FROM delta_scan('{path}')").fetchone()[0]
        print(f"  \u2705 {name:<40} {n:>6} rows")
    except Exception as e:
        print(f"  \u26a0\ufe0f  {name:<40} {e}")


📂 crm/
🦆 Found 7 Delta tables in lakehouse/:

  ✅ _run_logs                                     6 rows
  ✅ bronze_webshop_customers                     14 rows
  ✅ bronze_webshop_orders                        14 rows
  ✅ gold_webshop_daily_revenue                    0 rows
  ✅ gold_webshop_dim_customers                   11 rows
  ✅ silver_webshop_customers                     11 rows
  ✅ silver_webshop_orders_enriched                0 rows


---
## 9. 🔐 PII Masking — Contract-Defined Privacy

PII masking strategies are defined in each contract by the domain team:

| Field | Strategy | Effect |
|-------|----------|--------|
| `name` | `redact` | Replaced with `[REDACTED]` |
| `email` | `hash` | SHA-256 hashed (deterministic, join-safe) |
| `phone` | `partial` | Partially masked: `***-***-1234` |

Same rules apply whether running on DuckDB, Polars, or Spark.

In [55]:
import polars as pl

# ── Raw landing data (PII visible) ──────────────────────────────
raw_files = list((EXAMPLE_ROOT / "landing_crm" / "webshop" / "customers").rglob("*.csv"))
if raw_files:
    raw_df = pl.concat([pl.read_csv(str(f)) for f in raw_files])
    pii_cols = [c for c in ["customer_id", "name", "email", "phone"] if c in raw_df.columns]
    print("RAW Landing Data (PII visible):")
    display(raw_df.select(pii_cols).sort(pl.col("customer_id")).head(8))

# ── Materialized bronze (PII masked) ───────────────────────────
if "bronze_webshop_customers" in tables:
    print("MASKED Bronze Output (after PII masking):")
    display(
        con.sql(f"""
        SELECT customer_id, name, email, phone
        FROM delta_scan('{tables["bronze_webshop_customers"]}')
        ORDER BY customer_id asc
        LIMIT 8
    """).df()
    )

RAW Landing Data (PII visible):


customer_id,name,email,phone
str,str,str,str
"""CUST-040425""","""Michael Thomas""","""swood@example.org""","""897.490.8695x986"""
"""CUST-122317""","""Gerald Campbell""","""danieljones@example.org""","""451-640-5617x733"""
"""CUST-134957""","""Marcus Buchanan""","""mark72@example.org""","""350-694-5291"""
"""CUST-136360""","""Joseph Gregory""","""ramirezsusan@example.com""","""689-858-7147"""
"""CUST-166775""","""Daniel Smith""","""virginia59@example.org""",null
"""CUST-346222""","""Stephanie Dyer MD""","""wgreen@example.net""","""856.710.1363x964"""
"""CUST-413072""","""James Spence""","""barbara61@example.org""","""001-302-202-4418"""
"""CUST-428381""","""Joshua Atkins""","""noahroberts@example.org""","""890-712-8469"""


MASKED Bronze Output (after PII masking):


,customer_id,name,email,phone
0,CUST-040425,***REDACTED***,7b83e605b322bf458e0ef12ffd56421b593c7682d8518def4789a3ee0a1896be,***-***-5986
1,CUST-122317,***REDACTED***,46b591e30ece8be38e70b00806c8fc114b14977408fb58a3ddf5bda8b9afa4aa,***-***-7733
2,CUST-134957,***REDACTED***,7cbded7f0206a9c332f9efab36088a65237b16c824a64aeb6dc56cb7decc1a90,***-***-5291
3,CUST-136360,***REDACTED***,9e6b91bd8513a40e9f195969ce079b3b019e937c9a1e2357120e9b48dd984289,***-***-7147
4,CUST-166775,***REDACTED***,59ab1974c25b0d876ea098c5cfb385fd42a3cc8a68bafe08295219f9d5b1b8e2,None
5,CUST-346222,***REDACTED***,b5b5016bbf84d2209b74fe6910536d7e23b76722ea9c61be83181330855e38cd,***-***-3964
6,CUST-413072,***REDACTED***,e3f38c0a437b7a7e5ea0ee88a370f7f5223c0c1a945c7e0b7e77aeb138f9af4c,***-***-4418
7,CUST-428381,***REDACTED***,36bb28d2ba3e7fcf4c88bd84e48e2672e1c0167b44af7f3bf083017aaab42838,***-***-8469


In [56]:
# ── Silver: PII carried through JOIN, still masked ─────────────
if "silver_webshop_orders_enriched" in tables:
    print("Silver — Orders with masked customer PII:")
    display(
        con.sql(f"""
        SELECT order_id, customer_id, product_name,
               line_total, customer_name, customer_email,
               customer_segment, customer_country
        FROM delta_scan('{tables["silver_webshop_orders_enriched"]}')
        ORDER BY line_total DESC
        LIMIT 10
    """).df()
    )

Silver — Orders with masked customer PII:


,order_id,customer_id,product_name,line_total,customer_name,customer_email,customer_segment,customer_country


In [57]:
# ── Gold: Revenue by channel ───────────────────────────────────
if "gold_webshop_daily_revenue" in tables:
    print("Gold — Daily Revenue by Channel:")
    display(
        con.sql(f"""
        SELECT order_date, channel, customer_segment,
               total_orders, total_items, gross_revenue, avg_order_value
        FROM delta_scan('{tables["gold_webshop_daily_revenue"]}')
        ORDER BY gross_revenue DESC
        LIMIT 15
    """).df()
    )

Gold — Daily Revenue by Channel:


,order_date,channel,customer_segment,total_orders,total_items,gross_revenue,avg_order_value


---
## 10. 🛡️ Quarantine — Contract Violations

Records that violate contract rules are quarantined — never silently dropped.
The domain team gets visibility into data quality issues with full error context.

In [58]:
# ── Scan for quarantine output ─────────────────────────────────
import duckdb

q_con = duckdb.connect()
quarantine_found = False
QUARANTINE_DIR = OUTPUT / "_quarantine"
# Also check domain-level quarantine
if not QUARANTINE_DIR.exists():
    QUARANTINE_DIR = OUTPUT / "crm" / "_quarantine"

if QUARANTINE_DIR.exists() and QUARANTINE_DIR.is_dir():
    for delta_log in sorted(QUARANTINE_DIR.rglob("_delta_log")):
        table_dir = delta_log.parent
        path = str(table_dir).replace("\\", "/")
        label = str(table_dir.relative_to(QUARANTINE_DIR))
        try:
            n = q_con.sql(f"SELECT COUNT(*) FROM delta_scan('{path}')").fetchone()[0]
            print(f"\U0001f6e1\ufe0f {label}: {n} quarantined rows")
            if n > 0:
                display(q_con.sql(f"SELECT * FROM delta_scan('{path}') LIMIT 5").df())
            quarantine_found = True
        except Exception as e:
            print(f"\u26a0\ufe0f {label}: {e}")

q_con.close()

if not quarantine_found:
    print("\u2705 No quarantine tables found.")

🛡️ crm_silver_webshop_customers\silver_customers: 3 quarantined rows


,customer_id,name,email,phone,country,segment,signup_date,quarantine_state,quarantine_reprocessed,_lakelogic_errors,_lakelogic_source,_lakelogic_processed_at,_lakelogic_run_id,_lakelogic_created_at,_lakelogic_created_by
0,CUST-607407,***REDACTED***,5eee38f5494531d4097b2a6df7e80952bee14408954b271f96d76e6a785454a0,None,Australia,unknown,2026-01-22,active,False,[Rule failed: phone number present (phone IS NOT NULL)],C:\_Personal\_SaaS\lakelogic\examples\04_lakehouse_data_platform\lakehouse\crm\bronze_webshop_customers,2026-04-09T05:32:29+00:00,76a7d19b-eafb-4332-8b15-311bd6277070,2026-04-09T05:32:29+00:00,duckdb_demo
1,CUST-638571,***REDACTED***,c530f51210faff850b435f7b438b46574811a2cde171e075cc4d6db4fc364e57,None,Spain,at_risk,2026-03-05,active,False,[Rule failed: phone number present (phone IS NOT NULL)],C:\_Personal\_SaaS\lakelogic\examples\04_lakehouse_data_platform\lakehouse\crm\bronze_webshop_customers,2026-04-09T05:32:29+00:00,76a7d19b-eafb-4332-8b15-311bd6277070,2026-04-09T05:32:29+00:00,duckdb_demo
2,CUST-166775,***REDACTED***,59ab1974c25b0d876ea098c5cfb385fd42a3cc8a68bafe08295219f9d5b1b8e2,None,France,premium,2026-03-30,active,False,[Rule failed: phone number present (phone IS NOT NULL)],C:\_Personal\_SaaS\lakelogic\examples\04_lakehouse_data_platform\lakehouse\crm\bronze_webshop_customers,2026-04-09T05:32:29+00:00,76a7d19b-eafb-4332-8b15-311bd6277070,2026-04-09T05:32:29+00:00,duckdb_demo


🛡️ crm_silver_webshop_orders_enriched\silver_orders_enriched: 14 quarantined rows


,order_id,customer_id,order_date,product_name,category,quantity,unit_price,discount_pct,line_total,channel,status,customer_name,customer_email,customer_segment,customer_country,quarantine_state,quarantine_reprocessed,_lakelogic_errors,_lakelogic_source,_lakelogic_processed_at,_lakelogic_run_id,_lakelogic_created_at,_lakelogic_created_by
0,ORD-980635,CUST-897254,2026-03-01,Cooling Fan 120mm,Health,5364,5.31,192.0,-5.440222e+06,online,inactive,***REDACTED***,a58e600e5b031214da9ba1846e9679e9728a31e6082fcca93fcb7919601b6027,unknown,United Kingdom,active,False,[Rule failed: positive_line_total (line_total > 0)],C:\_Personal\_SaaS\lakelogic\examples\04_lakehouse_data_platform\lakehouse\crm\bronze_webshop_orders,2026-04-09T05:32:30+00:00,65bea0ed-0b5a-41ca-a4c6-dccea3e5a84b,2026-04-09T05:32:30+00:00,duckdb_demo
1,ORD-307966,CUST-772849,2026-01-17,Cooling Fan 120mm,Health,8581,37.70,268.0,-8.637549e+07,partner,active,***REDACTED***,e8509dc1616f53ff0265c9b047d199890bb5da0d43a5f095794b4a733c9546f6,unknown,Spain,active,False,[Rule failed: positive_line_total (line_total > 0)],C:\_Personal\_SaaS\lakelogic\examples\04_lakehouse_data_platform\lakehouse\crm\bronze_webshop_orders,2026-04-09T05:32:30+00:00,65bea0ed-0b5a-41ca-a4c6-dccea3e5a84b,2026-04-09T05:32:30+00:00,duckdb_demo
2,ORD-961429,CUST-136360,2026-02-02,Power Supply 500W,Sports,712,43.43,730.0,-2.254225e+07,partner,active,***REDACTED***,9e6b91bd8513a40e9f195969ce079b3b019e937c9a1e2357120e9b48dd984289,unknown,United Kingdom,active,False,[Rule failed: positive_line_total (line_total > 0)],C:\_Personal\_SaaS\lakelogic\examples\04_lakehouse_data_platform\lakehouse\crm\bronze_webshop_orders,2026-04-09T05:32:30+00:00,65bea0ed-0b5a-41ca-a4c6-dccea3e5a84b,2026-04-09T05:32:30+00:00,duckdb_demo
3,ORD-511340,CUST-428381,2026-02-14,Circuit Board,Health,7370,25.36,666.0,-1.242906e+08,social,active,***REDACTED***,36bb28d2ba3e7fcf4c88bd84e48e2672e1c0167b44af7f3bf083017aaab42838,at_risk,France,active,False,"[Rule failed: positive_line_total (line_total > 0), Rule failed: known_segment (customer_segment IN ('premium', 'standard', 'new', 'unknown'))]",C:\_Personal\_SaaS\lakelogic\examples\04_lakehouse_data_platform\lakehouse\crm\bronze_webshop_orders,2026-04-09T05:32:30+00:00,65bea0ed-0b5a-41ca-a4c6-dccea3e5a84b,2026-04-09T05:32:30+00:00,duckdb_demo
4,ORD-889570,CUST-134957,2026-02-18,Gadget Ultra,Books,3048,27.59,746.0,-6.265027e+07,mobile_app,active,***REDACTED***,7cbded7f0206a9c332f9efab36088a65237b16c824a64aeb6dc56cb7decc1a90,at_risk,South Africa,active,False,"[Rule failed: positive_line_total (line_total > 0), Rule failed: known_segment (customer_segment IN ('premium', 'standard', 'new', 'unknown'))]",C:\_Personal\_SaaS\lakelogic\examples\04_lakehouse_data_platform\lakehouse\crm\bronze_webshop_orders,2026-04-09T05:32:30+00:00,65bea0ed-0b5a-41ca-a4c6-dccea3e5a84b,2026-04-09T05:32:30+00:00,duckdb_demo


---
## 11. 📊 Pipeline Run Log

Every pipeline run is logged — providing full observability into what was processed,
how many rows passed/failed quality rules, and which contracts were executed.

The run log is stored as a Delta table in the lakehouse alongside the data products.
In production (Databricks), this maps to the `_run_logs` table in the domain schema.

In [59]:
# ── Inspect the pipeline run log ──────────────────────────────
import duckdb

log_con = duckdb.connect()
RUN_LOG_DIR = OUTPUT / "crm" / "_run_logs"
# if not RUN_LOG_DIR.exists():
#     RUN_LOG_DIR = OUTPUT / 'crm' / 'webshop' / '_run_logs'

if RUN_LOG_DIR.exists() and (RUN_LOG_DIR / "_delta_log").exists():
    log_path = str(RUN_LOG_DIR).replace("\\", "/")
    run_log_df = log_con.sql(f"""
        SELECT 
        pipeline_run_id,
        run_id, run_duration_seconds,
            dataset,
            data_layer,
            status,
            domain,
            system,
            counts_source,
            counts_total,
            counts_good,
            counts_quarantined,
            quarantine_ratio
        FROM delta_scan('{log_path}')
        ORDER BY timestamp, data_layer DESC
    """).df()

    print(f"📊 Pipeline Run Log — {len(run_log_df)} entries\n")
    display(run_log_df)
else:
    raise Exception("run log is emtpy, check config path for getting run logs")
log_con.close()

📊 Pipeline Run Log — 6 entries



,pipeline_run_id,run_id,run_duration_seconds,dataset,data_layer,status,domain,system,counts_source,counts_total,counts_good,counts_quarantined,quarantine_ratio
0,882287ee-2a2b-4035-8872-c7e7fcb9c3a2,8d7ba48e-0b01-4476-8e2b-77d362af1664,0.146263,bronze_webshop_orders,bronze,succeeded,crm,webshop,14,14,14,0,0.000000
1,882287ee-2a2b-4035-8872-c7e7fcb9c3a2,cc1d368c-a6f9-47c0-acb1-2b2f4cd8577e,0.137140,bronze_webshop_customers,bronze,succeeded,crm,webshop,14,14,14,0,0.000000
2,882287ee-2a2b-4035-8872-c7e7fcb9c3a2,76a7d19b-eafb-4332-8b15-311bd6277070,0.270361,silver_webshop_customers,silver,succeeded,crm,webshop,14,14,11,3,0.214286
3,882287ee-2a2b-4035-8872-c7e7fcb9c3a2,65bea0ed-0b5a-41ca-a4c6-dccea3e5a84b,0.684285,silver_webshop_orders_enriched,silver,succeeded,crm,webshop,14,14,0,14,1.000000
4,882287ee-2a2b-4035-8872-c7e7fcb9c3a2,58c79169-a6dd-4737-a601-301b5b142c85,0.138807,gold_webshop_dim_customers,gold,succeeded,crm,webshop,11,11,11,0,0.000000
5,882287ee-2a2b-4035-8872-c7e7fcb9c3a2,c7dde768-7582-4da7-875a-590cd312f065,0.756412,gold_webshop_daily_revenue,gold,succeeded,crm,webshop,0,0,0,0,NaN


---
## 12. 🎯 Summary

### Data Mesh Principles

| Principle | Implementation |
|-----------|---------------|
| **Domain Ownership** | CRM domain owns customers + orders across all layers |
| **Data as a Product** | Gold tables published with contracts, schemas, quality SLOs |
| **Self-Serve Platform** | `pip install lakelogic` — domain teams run pipelines independently |
| **Config-Driven Governance** | System registry provides defaults, contracts can override |

### Engine Portability

| Phase | Engine | When to Use |
|-------|--------|-------------|
| 🧪 Prototype | **DuckDB** | Instant feedback in a notebook — validate contracts, test rules |
| 🔬 Develop | **Polars** | Larger local datasets, CI/CD testing |
| 🏭 Production | **Spark / Databricks** | Cluster-scale, Unity Catalog, scheduled DABs |

**Same contracts. Same quality rules. Same PII masking. Any engine.**

### Try Next
- Change `ENGINE = 'polars'` — same contracts, different engine
- Add a quality rule and watch bad rows get quarantined
- Deploy to Databricks with `ENGINE = 'spark'` — zero contract changes
- Add a `marketing/` domain with its own `_system.yaml`

In [60]:
con.close()
print("Done! DuckDB connection closed.")

Done! DuckDB connection closed.
